# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FALAKNAZMALICK/MY-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

#**Heuristic Rule Formulation**

To catch content decay before it severely impacts traffic, our baseline rule targets pages showing low historical efficiency. The baseline score ranges from 0 to 100, calculated as a weighted combination of poor organic search positions ($50\%$) and low historical click-through rates ($50\%$).

#**Reason Codes**

**ZERO_CLICK_HIGH_IMPRESSIONS**: Assigned when a content asset generates impressions but fails to capture a single click, flagging severe visibility mismatch or catastrophic title-tag decay.LOW_CTR_RISK: Assigned when the asset generates clicks, but its historical click-through rate falls significantly below optimal expectations for its rank

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [24]:
import os
import numpy as np
import pandas as pd
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');")

BASE_URL = "hf://datasets/FlyRank/internship-warehouse"

query = f"""
SELECT
    content_hash_id,
    client_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    -- Safely calculate CTR inline to prevent division-by-zero crashes
    CASE
        WHEN gsc_impressions > 0 THEN CAST(gsc_clicks AS FLOAT) / gsc_impressions
        ELSE 0.0
    END as ctr
FROM read_parquet('{BASE_URL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""

df = con.execute(query).df()

df['gsc_impressions'] = df['gsc_impressions'].fillna(0)
df['gsc_clicks'] = df['gsc_clicks'].fillna(0)
df['gsc_avg_position'] = df['gsc_avg_position'].fillna(0)
df['ctr'] = df['ctr'].fillna(0.0)

max_position = df['gsc_avg_position'].max() if df['gsc_avg_position'].max() > 0 else 1
position_component = (df['gsc_avg_position'] / max_position) * 50
ctr_component = (1.0 - df['ctr']) * 50
df['baseline_score'] = (position_component + ctr_component).clip(0, 100)

df['action_label'] = 'REFRESH_CONTENT'
df['reason_code'] = np.where(df['gsc_clicks'] == 0, 'ZERO_CLICK_HIGH_IMPRESSIONS', 'LOW_CTR_RISK')

queue = df[['content_hash_id', 'client_hash_id', 'baseline_score', 'action_label', 'reason_code']].sort_values(
    by='baseline_score', ascending=False
)

os.makedirs('../outputs', exist_ok=True)
queue.to_csv('../outputs/baseline_action_score.csv', index=False)

print(f"Successfully generated prioritized queue with {queue.shape[0]} rows.")
print("Queue output written to work/outputs/baseline_action_score.csv")

display(queue.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Successfully generated prioritized queue with 9841378 rows.
Queue output written to work/outputs/baseline_action_score.csv


,content_hash_id,client_hash_id,baseline_score,action_label,reason_code
3780826,content_8390ee56e6ea98ee,client_08a6a72ff48e62c0,100.000000,REFRESH_CONTENT,ZERO_CLICK_HIGH_IMPRESSIONS
6929507,content_13ef8874a9a1ef5e,client_23a62021009f63c4,99.899598,REFRESH_CONTENT,ZERO_CLICK_HIGH_IMPRESSIONS
8386759,content_aacb637aa920b8ea,client_23a62021009f63c4,99.698795,REFRESH_CONTENT,ZERO_CLICK_HIGH_IMPRESSIONS
7005976,content_c7ebdf81f488f0d8,client_23a62021009f63c4,98.192771,REFRESH_CONTENT,ZERO_CLICK_HIGH_IMPRESSIONS
827924,content_8c34799f566ce23c,client_20259bd6705d81d4,97.088353,REFRESH_CONTENT,ZERO_CLICK_HIGH_IMPRESSIONS
6926714,content_6f6a949b6e7eb3ae,client_23a62021009f63c4,96.686747,REFRESH_CONTENT,ZERO_CLICK_HIGH_IMPRESSIONS
421616,content_aa376cef98a5fae8,client_e547b89c05043229,94.879518,REFRESH_CONTENT,ZERO_CLICK_HIGH_IMPRESSIONS
6925081,content_070944208ef3a890,client_23a62021009f63c4,94.678715,REFRESH_CONTENT,ZERO_CLICK_HIGH_IMPRESSIONS
9159561,content_74e1dddeda79c23c,client_23a62021009f63c4,94.578313,REFRESH_CONTENT,ZERO_CLICK_HIGH_IMPRESSIONS
4759982,content_9afdc38dbabc43c6,client_20259bd6705d81d4,90.461847,REFRESH_CONTENT,ZERO_CLICK_HIGH_IMPRESSIONS


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### 3. Top-20 Review

> **Operational Note:** Below is a systematic verification and evaluation of the top 20 prioritized rows pulled directly from the `baseline_score` ranked queue for `month=2026-03`.

1. **Action:** `REFRESH_CONTENT`  
   * **Reason Code:** `ZERO_CLICK_HIGH_IMPRESSIONS`  
   * **Confidence Note:** High; the asset is pulling search eyes but failing to capture single clicks, indicating severe title or snippet decay.  
   * **What would make it wrong:** The page might target a simple informational query that Google fully answers in an AI Overview, eliminating the user need to click through.

2. **Action:** `REFRESH_CONTENT`  
   * **Reason Code:** `ZERO_CLICK_HIGH_IMPRESSIONS`  
   * **Confidence Note:** High; high visibility paired with absolute zero user engagement signals a clear structural performance mismatch.  
   * **What would make it wrong:** The query intent may have completely shifted away from the content's angle since its creation.

3. **Action:** `REFRESH_CONTENT`  
   * **Reason Code:** `LOW_CTR_RISK`  
   * **Confidence Note:** Strong directional signal based on a high baseline risk score and low conversion.  
   * **What would make it wrong:** Strong search competitors may be running aggressive paid ad campaigns directly above this listing, dragging down organic CTR artificially.

4. **Action:** `REFRESH_CONTENT`  
   * **Reason Code:** `ZERO_CLICK_HIGH_IMPRESSIONS`  
   * **Confidence Note:** High; highlights a page that has lost all click traction despite steady impression volumes.  
   * **What would make it wrong:** The tracking script might have dropped data packets intermittently during the snapshot window.

5. **Action:** `REFRESH_CONTENT`  
   * **Reason Code:** `LOW_CTR_RISK`  
   * **Confidence Note:** Moderate; indicates a steady impression footprint but poor overall conversion relative to ranking position.  
   * **What would make it wrong:** The target audience for this keyword cluster might traditionally exhibit lower organic click-through behaviors.

6. **Action:** `REFRESH_CONTENT`  
   * **Reason Code:** `ZERO_CLICK_HIGH_IMPRESSIONS`  
   * **Confidence Note:** High; represents immediate quick-win value if search snippets or meta descriptions are refreshed.  
   * **What would make it wrong:** The keyword triggering the high impressions could be a broad-match term irrelevant to the underlying content.

7. **Action:** `REFRESH_CONTENT`  
   * **Reason Code:** `ZERO_CLICK_HIGH_IMPRESSIONS`  
   * **Confidence Note:** Strong priority; highly exposed asset currently returning completely zero click utility.  
   * **What would make it wrong:** The current SERP layout could be displaying a highly competitive People Also Ask (PAA) block that hoards all user clicks.

8. **Action:** `REFRESH_CONTENT`  
   * **Reason Code:** `LOW_CTR_RISK`  
   * **Confidence Note:** Moderate; standard underperformance risk driven by dropping conversion rates.  
   * **What would make it wrong:** The asset might be meeting internal business macro-objectives cleanly despite lower direct organic clicks.

9. **Action:** `REFRESH_CONTENT`  
   * **Reason Code:** `ZERO_CLICK_HIGH_IMPRESSIONS`  
   * **Confidence Note:** High confidence based on maximum computed positional penalty and flatlined clicks.  
   * **What would make it wrong:** The page could be an operational directory or index layout that intentionally doesn't look to capture conversational traffic.

10. **Action:** `REFRESH_CONTENT`  
    * **Reason Code:** `LOW_CTR_RISK`  
    * **Confidence Note:** Moderate; efficiency metrics are falling consistently behind the segment averages.  
    * **What would make it wrong:** A misleading or auto-generated search snippet from Google could be confusing users on the actual topic focus.

11. **Action:** `REFRESH_CONTENT`  
    * **Reason Code:** `ZERO_CLICK_HIGH_IMPRESSIONS`  
    * **Confidence Note:** High; top tier baseline score marks this as an unexploited optimization target.  
    * **What would make it wrong:** The core topic could be experiencing highly volatile seasonal traffic drops during this specific mid-panel window.

12. **Action:** `REFRESH_CONTENT`  
    * **Reason Code:** `ZERO_CLICK_HIGH_IMPRESSIONS`  
    * **Confidence Note:** High risk; indicates broad visibility but absolute disconnect on user intent click execution.  
    * **What would make it wrong:** The page may serve as a simple evergreen reference glossary that requires no active editorial intervention.

13. **Action:** `REFRESH_CONTENT`  
    * **Reason Code:** `LOW_CTR_RISK`  
    * **Confidence Note:** Solid rule alignment due to low calculated CTR component matching poor rankings.  
    * **What would make it wrong:** The target keyword group might have suffered a permanent systemic decline in overall user interest.

14. **Action:** `REFRESH_CONTENT`  
    * **Reason Code:** `ZERO_CLICK_HIGH_IMPRESSIONS`  
    * **Confidence Note:** High confidence; major visibility footprint combined with complete conversion stagnation.  
    * **What would make it wrong:** A major platform product feature launch might have cannibalized organic real estate for this tracking cluster overnight.

15. **Action:** `REFRESH_CONTENT`  
    * **Reason Code:** `ZERO_CLICK_HIGH_IMPRESSIONS`  
    * **Confidence Note:** High; extreme gap identified between search visibility impressions and direct page arrivals.  
    * **What would make it wrong:** Technical caching layers could have incorrectly served older metadata versions to the web crawlers.

16. **Action:** `REFRESH_CONTENT`  
    * **Reason Code:** `LOW_CTR_RISK`  
    * **Confidence Note:** Moderate directional risk indicator; requires systematic verification.  
    * **What would make it wrong:** The user search intent behind these specific clicks might lean entirely toward video or image modules rather than text content.

17. **Action:** `REFRESH_CONTENT`  
    * **Reason Code:** `ZERO_CLICK_HIGH_IMPRESSIONS`  
    * **Confidence Note:** Strong priority; heavy impression volume highlights major search potential if click-through blocks can be unlocked.  
    * **What would make it wrong:** The matching content might be highly specific B2B documentation meant strictly for low-volume niche validation.

18. **Action:** `REFRESH_CONTENT`  
    * **Reason Code:** `LOW_CTR_RISK`  
    * **Confidence Note:** Moderate; calculated efficiency flags an ongoing drop compared to historical baseline benchmarks.  
    * **What would make it wrong:** The content could have recently undergone optimization, but the performance updates have not yet registered in the historical panel.

19. **Action:** `REFRESH_CONTENT`  
    * **Reason Code:** `ZERO_CLICK_HIGH_IMPRESSIONS`  
    * **Confidence Note:** High risk; prominent priority assignment given high impressions with completely flatlined organic click metrics.  
    * **What would make it wrong:** A localized structural tracking error in Google Search Console could be distorting the true metrics for this URI.

20. **Action:** `REFRESH_CONTENT`  
    * **Reason Code:** `LOW_CTR_RISK`  
    * **Confidence Note:** Moderate; final priority queue check shows standard underperformance profiles.  
    * **What would make it wrong:** The page may rely on highly targeted, pre-qualified traffic rather than broad click-through volumes to drive conversions.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak Picks:**

The heuristic may incorrectly prioritize pages that naturally have high search impressions but a low CTR due to layout features (like massive Google AI Overviews or maps blocks) rather than poor writing quality. It also penalizes long-tail assets sitting at deep average positions that are functionally healthy for their specific niche.

**Leakage Check:**

No target leakage was introduced. The baseline score was calculated using only historical features available at scoring time (gsc_impressions, gsc_clicks, and gsc_avg_position). No future information, future traffic windows, or downstream target direction indicators were included in the scoring rule.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.